# E035 — offline full hidden-test E026 + guarded CAFormer

This code-submission version reruns both arms on the hidden test set. It first executes
the exact E026 `0.927` single-CoAtNet safe-window inference, then loads the exact E034
CAFormer-B36 backbone from an attached Kaggle Model and the SHA-locked two-head E035
checkpoint. Four risk targets remain exactly E026; only the prelocked eight targets may
blend. Internet access is disabled.


In [ ]:
#!/usr/bin/env python3
"""Knee MRI: twelve findings with series-safe 2.5D windows

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study, resampled only from windows that stay inside one acquisition series. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
IMG = 336
CROP_MM = 140.0
# 64 slices per study instead of 44, same proportions. Must match the corpus the weights
# were trained on (knee_corpus_v4.py).
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
         ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)                     # 64
K_EVAL = 42   # every window position the volume holds, not an evenly spaced subset
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [
    {"file": "raptor_ft_coatnet_v4_full.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    """Resample centers only where the full 2.5D window stays in one series slot.

    ``build_study`` assigns every acquisition to one fixed contiguous slot and never
    reuses a SeriesInstanceUID.  Requiring all three mask values and keeping the
    center inside one slot is therefore equivalent to E025's same-SeriesInstanceUID
    predicate, without changing pixels, weights, normalization, or model code.
    """
    safe = []
    cursor = 0
    for _, _, slot_depth in SLOTS:
        lo = cursor
        hi = min(cursor + int(slot_depth), D)
        safe.extend(
            center
            for center in range(lo + 1, hi - 1)
            if bool(mask[center - 1:center + 2].all())
        )
        cursor += int(slot_depth)
        if cursor >= D:
            break

    # Preserve E020's no-study-dropped contract for a pathological all-missing
    # study.  Normal studies use only the series-safe branch above.
    if not safe:
        valid = np.where(mask > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        lo, hi = int(valid.min()), int(valid.max())
        safe = [
            center
            for center in range(lo + 1, hi)
            if center - 1 >= lo and center + 1 <= hi
        ]
        if not safe:
            safe = [max(1, min((lo + hi) // 2, D - 2))]

    indexes = np.linspace(0, len(safe) - 1, k).round().astype(int)
    return [safe[index] for index in indexes]

def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(device)
    use_cuda = device != "cpu" and str(device).startswith("cuda")
    if use_cuda:
        # fp16 conv on T4 is fully cuDNN-supported (bf16 is NOT -> "no engine").
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
        except RuntimeError:
            # fp32 always has a Turing conv engine; slower but never drops a study.
            torch.cuda.empty_cache()
            o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
    o = torch.sigmoid(model(x).float())
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (knee_corpus_v2.py, SPAN_LO/SPAN_HI).
        n = len(files); lo, hi = int(n * 0.06), int(n * 0.94) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs = []; pss = []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(read_px(fp)); pss.append(ps)
            except Exception:
                arrs.append(None); pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask


# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower():
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    reader = _make_reader()
    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]

    # SEQUENTIAL ARMS (the OOM fix): only ONE model is resident at a time, so peak system RAM ==
    # one model == the single-arm champion's footprint (which graded fine at 0.879). Holding both
    # arms simultaneously OOM'd system RAM on the full hidden test. Each study is re-preprocessed
    # per arm (build_study is cheap vs inference) and every per-study buffer is freed. Same models,
    # same windowing, same rank-mean blend -> identical 0.8893 result, just serialized.
    for a, arm in enumerate(ARMS):
        wp = find_weight_file(arm["file"])
        model, res = load_model(wp, arm["arch"], arm["res"], dev)
        print(f"[arm {a}] loaded {arm['file']} | res {res} | {time.time()-t0:.0f}s", flush=True)
        for i, sid in enumerate(test_ids):
            try:
                vol, mask = build_study(sid, SER, tsdir, reader)
                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)
                arm_probs[a][i] = infer_probs(model, xw, dev)
                del vol, mask, xw
            except Exception as e:
                print(f"  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            if (i + 1) % 100 == 0 or i + 1 == N:
                print(f"  [arm {a}] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
        del model
        gc.collect()
        if str(dev).startswith("cuda"):
            torch.cuda.empty_cache()
        print(f"[arm {a}] done + freed | {time.time()-t0:.0f}s", flush=True)

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path

import hashlib

import cv2
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = list(LAB)
HEAD_DIM = 512
K_WINDOWS = K = 24
RES = 384
IMG = 336
CROP_MM = 140.0
SLOTS = [
    ("sagittal_fluid", "Sagittal", 1, 18),
    ("sagittal_other", "Sagittal", 0, 14),
    ("coronal_fluid", "Coronal", 1, 12),
    ("coronal_other", "Coronal", 0, 8),
    ("axial", "Axial", -1, 12),
]
MAXS = sum(slot[3] for slot in SLOTS)
CAFO_ARCH = "caformer_b36.sail_in22k_ft_in1k_384"
CAFO_FEATURE_DIM = 768
EXPECTED_CAFO_STATE_SHA256 = "ed2ace0e00efcfe9e131a8e83372194817fdf69860ad6525258b908222a4c6ae"
HEADS_SHA256 = "29ba27966da2328372b2f08389ab5cfa566cb7667b83b687a0402530087d2704"
HEADS_TRAIN_IDS_SHA256 = "8b92e40d67ca3b52da820b8d4b9049e2a276307ba208fe80667f100b91b68020"
FULL_HEAD_SEEDS = [35001, 35002]
GUARDED_ALPHA = np.asarray([
    0.50, 0.00, 0.75, 0.75, 0.50, 0.75,
    0.75, 0.50, 0.00, 0.50, 0.00, 0.00,
], dtype=np.float32)
PROTECTED_TARGETS = ["MCL", "Synovitis", "Contusion", "Fracture"]
def find_root():
    for root in [INPUT / 'competitions/rsna-knee-abnormality-detection', INPUT / 'rsna-knee-abnormality-detection']:
        if (root / 'train_series.csv').is_file():
            return root
    raise FileNotFoundError('competition mount not found')


def dataset_roots():
    roots = list(INPUT.glob('*')) + list(INPUT.glob('datasets/*/*'))
    return [root for root in roots if root.is_dir() and 'competitions' not in root.parts and (root.name != 'datasets')]


def find_input_file(name, preferred=()):
    hits = []
    for root in dataset_roots():
        direct = root / name
        if direct.is_file():
            hits.append(direct)
        hits.extend((path for path in root.glob(f'*/{name}') if path.is_file()))
    hits.sort(key=lambda path: (not any((token in str(path).lower() for token in preferred)), len(str(path))))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def fluid(value):
    try:
        return int(float(value))
    except Exception:
        return 0


def pick_series(rows, plane, preferred_fluid, used):
    candidates = [row for row in rows if str(row.get('Anatomical_Plane', '')).lower() == plane.lower() and str(row['SeriesInstanceUID']) not in used]
    if preferred_fluid in (0, 1):
        preferred = [row for row in candidates if fluid(row.get('Fluid_Sensitive', 0)) == preferred_fluid]
        if preferred:
            candidates = preferred
    return candidates[0] if candidates else None


def ordered_records(directory: Path):
    records = []
    for path in sorted(directory.glob('*.dcm')):
        try:
            header = pydicom.dcmread(str(path), stop_before_pixels=True)
            iop = getattr(header, 'ImageOrientationPatient', None)
            ipp = getattr(header, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and (len(iop) == 6):
                normal = np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:], float))
                position = float(np.dot(np.asarray(ipp, float), normal))
            else:
                position = float(getattr(header, 'InstanceNumber', 0) or 0)
            spacing = getattr(header, 'PixelSpacing', [0.5, 0.5])
            records.append((position, path, float(spacing[0])))
        except Exception:
            continue
    return sorted(records, key=lambda row: row[0])


def pixels(path):
    dicom = pydicom.dcmread(str(path))
    array = apply_modality_lut(dicom.pixel_array, dicom).astype(np.float32)
    if str(getattr(dicom, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        array = array.max() - array
    return array


def crop(array01, spacing):
    height, width = array01.shape
    half = max(2, int(round(CROP_MM / max(spacing, 0.001) / 2)))
    cy, cx = (height // 2, width // 2)
    value = array01[max(0, cy - half):min(height, cy + half), max(0, cx - half):min(width, cx + half)]
    return cv2.resize(value if value.size else array01, (IMG, IMG), interpolation=cv2.INTER_AREA)


def build_study(uid):
    rows = series_by_uid.get(uid, [])
    volume = np.zeros((64, IMG, IMG), np.uint8)
    mask = np.zeros(64, np.uint8)
    provenance, used, cursor = ([], set(), 0)
    for slot_index, (slot, plane, preferred_fluid, count) in enumerate(SLOTS):
        row = pick_series(rows, plane, preferred_fluid, used)
        if row is None:
            for slot_position in range(count):
                provenance.append({'StudyInstanceUID': uid, 'index': cursor, 'slot': slot, 'slot_index': slot_index, 'slot_position': slot_position, 'series': None, 'raw_index': None, 'raw_count': 0, 'valid': 0})
                cursor += 1
            continue
        series_uid = str(row['SeriesInstanceUID'])
        used.add(series_uid)
        records = ordered_records(TRAIN_DCM / uid / series_uid)
        count_raw = len(records)
        lo = int(count_raw * 0.06)
        hi = max(int(count_raw * 0.94) - 1, lo)
        picks = np.linspace(lo, hi, count).round().astype(int) if count_raw > 1 else np.zeros(count, int)
        arrays = []
        for raw_index in picks:
            try:
                arrays.append(pixels(records[min(int(raw_index), count_raw - 1)][1]))
            except Exception:
                arrays.append(None)
        valid_arrays = [array for array in arrays if array is not None]
        if valid_arrays:
            values = np.concatenate([array.reshape(-1) for array in valid_arrays])
            p02, p98 = np.percentile(values, [2, 98])
        else:
            p02, p98 = (0.0, 1.0)
        for slot_position, (raw_index, array) in enumerate(zip(picks, arrays)):
            valid = 0
            if array is not None:
                record = records[min(int(raw_index), count_raw - 1)]
                array01 = np.clip((array - p02) / (p98 - p02 + 1e-06), 0, 1)
                volume[cursor] = np.round(crop(array01, record[2]) * 255).astype(np.uint8)
                mask[cursor] = 1
                valid = 1
            provenance.append({'StudyInstanceUID': uid, 'index': cursor, 'slot': slot, 'slot_index': slot_index, 'slot_position': slot_position, 'series': series_uid, 'raw_index': int(raw_index), 'raw_count': count_raw, 'valid': valid})
            cursor += 1
    assert cursor == 64
    return (volume, mask, pd.DataFrame(provenance))


def safe_candidates(mask, provenance):
    candidates = []
    for center in range(1, len(mask) - 1):
        tri = provenance.iloc[[center - 1, center, center + 1]]
        if bool(mask[center - 1:center + 2].all()) and bool(tri['series'].notna().all()) and (int(tri['series'].nunique()) == 1):
            candidates.append(center)
    return candidates


def resample_centers(candidates, count=K):
    if not candidates:
        return []
    indexes = np.linspace(0, len(candidates) - 1, count).round().astype(int)
    return [int(candidates[index]) for index in indexes]


def sha256_file(path: Path, chunk_size=8 << 20):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while (chunk := handle.read(chunk_size)):
            digest.update(chunk)
    return digest.hexdigest()


def module_sha256(module):
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        digest.update(name.encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()


class TargetWindowMIL(nn.Module):

    def __init__(self, input_dim, hidden_dim=HEAD_DIM, targets=len(TARGETS), dropout=0.15):
        super().__init__()
        self.projection = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.attention = nn.Sequential(nn.Linear(hidden_dim, 256), nn.Tanh(), nn.Dropout(dropout), nn.Linear(256, targets))
        self.classifier_weight = nn.Parameter(torch.empty(targets, hidden_dim))
        self.classifier_bias = nn.Parameter(torch.zeros(targets))
        nn.init.trunc_normal_(self.classifier_weight, std=0.02)

    def forward(self, features, return_attention=False):
        hidden = self.projection(features)
        attention = torch.softmax(self.attention(hidden), dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', attention, hidden)
        logits = (pooled * self.classifier_weight.unsqueeze(0)).sum(-1) + self.classifier_bias
        if return_attention:
            return (logits, attention.transpose(1, 2))
        return logits



def find_caformer_checkpoint() -> Path:
    search_roots = [
        root
        for root in INPUT.iterdir()
        if root.is_dir()
        and (root.name.lower() == "models" or "caformer" in root.name.lower())
    ]
    candidates = sorted(
        {
            path.resolve()
            for root in search_roots
            for path in root.rglob("model.safetensors")
            if "caformer" in str(path).lower()
        }
    )
    if len(candidates) != 1:
        raise RuntimeError(
            "expected one CAFormer safetensors file; "
            f"search_roots={search_roots}, candidates={candidates}"
        )
    return candidates[0]


def windows_from_study(uid: str):
    volume, mask, provenance = build_study(uid)
    candidates = safe_candidates(mask, provenance)
    centers = resample_centers(candidates, count=K_WINDOWS)
    if len(centers) != K_WINDOWS:
        raise RuntimeError(f"{uid}: safe-window contract failed")
    windows = np.stack([
        volume[[center - 1, center, center + 1]].astype(np.float32) / 255.0
        for center in centers
    ])
    tensor = F.interpolate(
        torch.from_numpy(windows),
        size=(RES, RES),
        mode="bilinear",
        align_corners=False,
    )
    return tensor, centers


root = find_root()
TRAIN_DCM = root / "test_series"
if not TRAIN_DCM.is_dir():
    TRAIN_DCM = root / "test_images"
if not TRAIN_DCM.is_dir():
    raise FileNotFoundError("test DICOM directory")
test_series = pd.read_csv(
    root / "test_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)
test_series["StudyInstanceUID"] = test_series["StudyInstanceUID"].astype(str)
test_series["SeriesInstanceUID"] = test_series["SeriesInstanceUID"].astype(str)
series_by_uid = {
    uid: frame.to_dict("records")
    for uid, frame in test_series.groupby("StudyInstanceUID")
}

baseline_path = WORK / "submission.csv"
if not baseline_path.is_file():
    raise FileNotFoundError("E026 baseline submission.csv")
baseline = pd.read_csv(baseline_path, dtype={"StudyInstanceUID": str})
if list(baseline.columns) != ["StudyInstanceUID"] + TARGETS:
    raise RuntimeError("E026 baseline schema mismatch")
test_ids = baseline["StudyInstanceUID"].astype(str).tolist()
if not set(test_ids).issubset(set(test_series["StudyInstanceUID"])):
    raise RuntimeError("baseline/test-series UID mismatch")
baseline_values = baseline[TARGETS].to_numpy(np.float64)
baseline_sha256 = sha256_file(baseline_path)

heads_path = find_input_file("e035_caformer_heads.pt", ("e035-assets",))
if sha256_file(heads_path) != HEADS_SHA256:
    raise RuntimeError("E035 head checkpoint SHA mismatch")
heads_payload = torch.load(heads_path, map_location="cpu", weights_only=False)
if heads_payload.get("architecture") != CAFO_ARCH:
    raise RuntimeError("head architecture mismatch")
if heads_payload.get("targets") != TARGETS:
    raise RuntimeError("head target mismatch")
if heads_payload.get("head_seeds") != FULL_HEAD_SEEDS:
    raise RuntimeError("head seed mismatch")
if heads_payload.get("train_ids_sha256") != HEADS_TRAIN_IDS_SHA256:
    raise RuntimeError("head train-ID mismatch")
if len(heads_payload.get("head_state_dicts", [])) != 2:
    raise RuntimeError("head count mismatch")
heads = []
for state_dict in heads_payload["head_state_dicts"]:
    head = TargetWindowMIL(input_dim=CAFO_FEATURE_DIM)
    head.load_state_dict(state_dict, strict=True)
    heads.append(head.to(DEVICE).eval())
del heads_payload

caformer_path = find_caformer_checkpoint()
caformer_model = timm.create_model(
    CAFO_ARCH,
    pretrained=False,
    num_classes=0,
    in_chans=3,
    global_pool="avg",
)
timm.models.load_checkpoint(caformer_model, str(caformer_path), strict=False)
caformer_state_sha256 = module_sha256(caformer_model)
if caformer_state_sha256 != EXPECTED_CAFO_STATE_SHA256:
    raise RuntimeError(f"CAFormer state SHA mismatch: {caformer_state_sha256}")
if int(caformer_model.num_features) != CAFO_FEATURE_DIM:
    raise RuntimeError("CAFormer feature dimension mismatch")
config = timm.data.resolve_model_data_config(caformer_model)
mean = torch.tensor(config["mean"], dtype=torch.float32).view(3, 1, 1)
std = torch.tensor(config["std"], dtype=torch.float32).view(3, 1, 1)
caformer_model = caformer_model.to(DEVICE).eval()

caformer_probabilities = np.full((len(test_ids), len(TARGETS)), 0.5, np.float32)
failures = []
started = time.time()
with torch.inference_mode():
    for study_index, uid in enumerate(test_ids):
        try:
            tensor, _centers = windows_from_study(uid)
            tensor = (tensor - mean) / std
            feature_chunks = []
            for start in range(0, K_WINDOWS, 6):
                values = tensor[start:start + 6].to(DEVICE, non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                    encoded = caformer_model(values)
                feature_chunks.append(encoded.float())
            features = torch.cat(feature_chunks, dim=0).unsqueeze(0)
            if tuple(features.shape) != (1, K_WINDOWS, CAFO_FEATURE_DIM):
                raise RuntimeError(f"bad feature shape {tuple(features.shape)}")
            probabilities = []
            for head in heads:
                probabilities.append(torch.sigmoid(head(features)).float().cpu().numpy()[0])
            caformer_probabilities[study_index] = np.mean(np.stack(probabilities), axis=0)
            del tensor, feature_chunks, features
        except Exception as error:
            failures.append({
                "StudyInstanceUID": uid,
                "error_type": type(error).__name__,
                "error": str(error),
            })
        if (study_index + 1) % 50 == 0 or study_index + 1 == len(test_ids):
            print("CAFormer hidden inference", study_index + 1, "/", len(test_ids),
                  "seconds", round(time.time() - started, 1), flush=True)

if not np.isfinite(caformer_probabilities).all():
    raise RuntimeError("non-finite CAFormer probabilities")
del caformer_model, heads
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

caformer_ranks = rankpct(caformer_probabilities)
blended_values = baseline_values.copy()
for target_index, alpha in enumerate(GUARDED_ALPHA):
    blended_values[:, target_index] = (
        (1.0 - float(alpha)) * baseline_values[:, target_index]
        + float(alpha) * caformer_ranks[:, target_index]
    )
uid_to_index = {uid: index for index, uid in enumerate(test_ids)}
for failure in failures:
    blended_values[uid_to_index[failure["StudyInstanceUID"]]] = baseline_values[
        uid_to_index[failure["StudyInstanceUID"]]
    ]
if not np.isfinite(blended_values).all() or ((blended_values < 0) | (blended_values > 1)).any():
    raise RuntimeError("invalid blended values")

submission = pd.DataFrame(blended_values.astype(np.float32), columns=TARGETS)
submission.insert(0, "StudyInstanceUID", test_ids)
sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
if list(submission.columns) != list(sample.columns):
    raise RuntimeError("submission/sample schema mismatch")
if submission["StudyInstanceUID"].tolist() != sample["StudyInstanceUID"].tolist():
    raise RuntimeError("submission/sample UID order mismatch")

target_audit = []
for target_index, target in enumerate(TARGETS):
    baseline_order = baseline_values[:, target_index].argsort(kind="mergesort").argsort(kind="mergesort")
    blend_order = blended_values[:, target_index].argsort(kind="mergesort").argsort(kind="mergesort")
    target_audit.append({
        "target": target,
        "alpha": float(GUARDED_ALPHA[target_index]),
        "numeric_changed": bool(not np.array_equal(baseline_values[:, target_index], blended_values[:, target_index])),
        "ranking_changed": bool(not np.array_equal(baseline_order, blend_order)),
        "protected_exact": bool(
            target not in PROTECTED_TARGETS
            or np.array_equal(baseline_values[:, target_index], blended_values[:, target_index])
        ),
    })
audit = pd.DataFrame(target_audit)
if not audit.loc[audit["target"].isin(PROTECTED_TARGETS), "protected_exact"].all():
    raise RuntimeError("protected target changed")
changed_rank_targets = audit.loc[audit["ranking_changed"], "target"].tolist()

submission_path = WORK / "submission.csv"
submission.to_csv(submission_path, index=False)
audit.to_csv(WORK / "e035_target_audit.csv", index=False)
pd.DataFrame(failures, columns=["StudyInstanceUID", "error_type", "error"]).to_csv(
    WORK / "e035_hidden_failures.csv", index=False
)
decision = "READY_TO_SUBMIT_E035" if changed_rank_targets else "NO_RANK_CHANGE_DO_NOT_SUBMIT"
summary = {
    "experiment": "E035_OFFLINE_FULL_HIDDEN_INFERENCE",
    "decision": decision,
    "anchor": {
        "kernel": "E026 exact full hidden inference",
        "public_score": 0.927,
        "sha256": baseline_sha256,
    },
    "training": {
        "study_count": 640,
        "train_ids_sha256": HEADS_TRAIN_IDS_SHA256,
        "gold_excluded": True,
        "head_seeds": FULL_HEAD_SEEDS,
        "checkpoint_sha256": HEADS_SHA256,
    },
    "test": {
        "study_count": len(test_ids),
        "valid_encoded": len(test_ids) - len(failures),
        "failure_count": len(failures),
    },
    "guard": dict(zip(TARGETS, [float(value) for value in GUARDED_ALPHA])),
    "protected_targets": PROTECTED_TARGETS,
    "changed_rank_targets": changed_rank_targets,
    "caformer_state_sha256": caformer_state_sha256,
    "submission_sha256": sha256_file(submission_path),
    "internet_required": False,
}
(WORK / "e035_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(submission.to_string(index=False))
